In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("Merged_Raw_Dataset.csv") 

C:\Users\ASUS\AppData\Local\Temp\ipykernel_9984\2021997515.py:1: DtypeWarning: Columns (0: Overall_Score, 1: AGE, 2: Academic Reputation Score, 3: Employer Reputation Score, 4: Faculty Student Score, 5: Citations per Faculty Score, 6: International Faculty Score, 7: International Students Score, 8: International Research Network Score, 9: Employment Outcomes Score, 10: Sustainability Score, 11: international) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Merged_Raw_Dataset.csv")


In [1]:
import os
for f in os.listdir():
    print(f)

.git
CleanedData.ipynb
Data2
Education day 2.csv
MergeDatasets.ipynb
Merged_Raw_Dataset.csv
README.md


In [4]:
print("Shape:", df.shape)


Shape: (17661, 79)


In [5]:
print(df.dtypes)


RANK_2025              str
RANK_2024              str
university_name        str
Location               str
Region                 str
                    ...   
hici               float64
ns                 float64
pub                float64
pcp                float64
year_y             float64
Length: 79, dtype: object


In [6]:
df.head()


,RANK_2025,RANK_2024,university_name,Location,Region,SIZE_x,FOCUS_x,RES._x,STATUS_x,Academic_Reputation_Score,...,world_rank_y,national_rank,total_score_y,alumni,award,hici,ns,pub,pcp,year_y
0,306,336,aalborg university,Denmark,Europe,L,FC,VH,A,19.9,...,401-500,5,NaN,0.0,0.0,12.2,3.4,27.6,15.2,2014.0
1,306,336,aalborg university,Denmark,Europe,L,FC,VH,A,19.9,...,301-400,5,NaN,0.0,0.0,11.2,4.6,30.4,16.8,2015.0
2,306,336,aalborg university,Denmark,Europe,L,FC,VH,A,19.9,...,401-500,5,NaN,0.0,0.0,12.2,3.4,27.6,15.2,2014.0
3,306,336,aalborg university,Denmark,Europe,L,FC,VH,A,19.9,...,301-400,5,NaN,0.0,0.0,11.2,4.6,30.4,16.8,2015.0
4,306,336,aalborg university,Denmark,Europe,L,FC,VH,A,19.9,...,401-500,5,NaN,0.0,0.0,12.2,3.4,27.6,15.2,2014.0


In [7]:
print("Full duplicate rows:", df.duplicated().sum())


Full duplicate rows: 0


In [8]:
print("Duplicate university_name entries:", df.duplicated(subset=['university_name']).sum())


Duplicate university_name entries: 15364


In [9]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Dropped {before-after} exact duplicate rows")

Dropped 0 exact duplicate rows


In [10]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace('ï»¿', '', regex=False)
    .str.replace(' ', '_')
    .str.replace('.', '', regex=False)
    .str.lower()
)
print(df.columns.tolist())

['rank_2025', 'rank_2024', 'university_name', 'location', 'region', 'size_x', 'focus_x', 'res_x', 'status_x', 'academic_reputation_score', 'academic_reputation_rank', 'employer_reputation_score', 'employer_reputation_rank', 'faculty_student_score', 'faculty_student_rank', 'citations_per_faculty_score', 'citations_per_faculty_rank', 'international_faculty_score', 'international_faculty_rank', 'international_students_score', 'international_students_rank', 'international_research_network_score', 'international_research_network_rank', 'employment_outcomes_score', 'employment_outcomes_rank', 'sustainability_score', 'sustainability_rank', 'overall_score', '2024_rank', '2023_rank', 'country_code', 'country', 'size_y', 'focus_y', 'res_y', 'age', 'status_y', 'academic_reputation_score', 'academic_reputation_rank', 'employer_reputation_score', 'employer_reputation_rank', 'faculty_student_score', 'faculty_student_rank', 'citations_per_faculty_score', 'citations_per_faculty_rank', 'international_f

In [11]:
cols = list(df.columns)


In [12]:
qs2025_start = cols.index('status_x') + 1
qs2025_end = cols.index('overall_score') + 1

In [14]:
qs2024_start = cols.index('status_y') + 1
qs2024_end = qs2024_start + (qs2025_end - qs2025_start)  # same number of metrics

for i in range(qs2025_start, qs2025_end):
    cols[i] = cols[i] + '_2025'

for i in range(qs2024_start, qs2024_end):
    cols[i] = cols[i] + '_2024'

df.columns = cols

# Fix duplicate 'country' columns similarly
country_positions = [i for i, c in enumerate(cols) if c == 'country']
print("country column positions:", country_positions)


country column positions: [31, 57]


In [15]:
print(df.columns[df.columns.duplicated()].tolist())

['country']


In [ ]:
cols = list(df.columns)

country_positions = [i for i, c in enumerate(cols) if c == 'country']
print("Positions:", country_positions)

# First 'country' sits in the QS block (next to country_code) ...this is the QS-reported country
# Second 'country' sits in the timesData block...this is THE-reported country
cols[country_positions[0]] = 'country_qs'
cols[country_positions[1]] = 'country_times'

df.columns = cols

# Confirm no duplicates remain anywhere
print(df.columns[df.columns.duplicated()].tolist())

Positions: [31, 57]
[]


In [17]:
df = df.rename(columns={
    'size_x': 'qs_size_2025', 'focus_x': 'qs_focus_2025', 'res_x': 'qs_research_2025', 'status_x': 'qs_status_2025',
    'size_y': 'qs_size_2024', 'focus_y': 'qs_focus_2024', 'res_y': 'qs_research_2024', 'status_y': 'qs_status_2024',
    'world_rank_x': 'times_world_rank', 'total_score_x': 'times_total_score',
    'world_rank_y': 'shanghai_world_rank', 'total_score_y': 'shanghai_total_score',
    'year_x': 'times_year', 'year_y': 'shanghai_year',
    '2024_rank': 'times_2024_rank', '2023_rank': 'times_2023_rank'
})
print(df.columns.tolist())

['rank_2025', 'rank_2024', 'university_name', 'location', 'region', 'qs_size_2025', 'qs_focus_2025', 'qs_research_2025', 'qs_status_2025', 'academic_reputation_score_2025', 'academic_reputation_rank_2025', 'employer_reputation_score_2025', 'employer_reputation_rank_2025', 'faculty_student_score_2025', 'faculty_student_rank_2025', 'citations_per_faculty_score_2025', 'citations_per_faculty_rank_2025', 'international_faculty_score_2025', 'international_faculty_rank_2025', 'international_students_score_2025', 'international_students_rank_2025', 'international_research_network_score_2025', 'international_research_network_rank_2025', 'employment_outcomes_score_2025', 'employment_outcomes_rank_2025', 'sustainability_score_2025', 'sustainability_rank_2025', 'overall_score_2025', 'times_2024_rank', 'times_2023_rank', 'country_code', 'country_qs', 'qs_size_2024', 'qs_focus_2024', 'qs_research_2024', 'age', 'qs_status_2024', 'academic_reputation_score_2024', 'academic_reputation_rank_2024', 'empl

In [18]:
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 0])

shanghai_total_score                73.744409
overall_score_2025                  46.175188
international_faculty_score_2024    35.037654
international_faculty_rank_2024     35.037654
international_faculty_score_2025    34.420474
                                      ...    
alumni                              12.592718
national_rank                       12.592718
shanghai_world_rank                 12.587056
shanghai_year                       12.587056
university_name                      0.005662
Length: 79, dtype: float64


In [19]:
pd.set_option('display.max_rows', None)
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 0])

shanghai_total_score                         73.744409
overall_score_2025                           46.175188
international_faculty_score_2024             35.037654
international_faculty_rank_2024              35.037654
international_faculty_score_2025             34.420474
international_faculty_rank_2025              34.420474
sustainability_score_2024                    34.363853
sustainability_rank_2024                     34.363853
qs_research_2024                             34.318555
times_2023_rank                              34.193987
international_students_score_2024            34.182662
international_students_rank_2024             34.182662
qs_status_2024                               33.967499
international_students_rank_2025             33.888228
international_students_score_2025            33.888228
citations_per_faculty_rank_2024              33.865580
employment_outcomes_score_2024               33.865580
employment_outcomes_rank_2024                33.865580
faculty_st

In [20]:
def get_source(row):
    sources = []
    if pd.notnull(row['rank_2025']) or pd.notnull(row['rank_2024']):
        sources.append('QS')
    if pd.notnull(row['times_world_rank']):
        sources.append('Times')
    if pd.notnull(row['shanghai_world_rank']):
        sources.append('Shanghai')
    return '+'.join(sources) if sources else 'Unknown'

df['data_source'] = df.apply(get_source, axis=1)
print(df['data_source'].value_counts())

data_source
QS+Times+Shanghai    9987
Times+Shanghai       3573
Shanghai             1517
QS                    972
Times                 603
QS+Times              414
QS+Shanghai           361
Unknown               234
Name: count, dtype: int64


In [21]:
before = df.shape[0]
df = df[df['data_source'] != 'Unknown']
after = df.shape[0]
print(f"Dropped {before-after} rows with no identifiable ranking source")
print("New shape:", df.shape)

Dropped 234 rows with no identifiable ranking source
New shape: (17427, 80)


In [22]:
# Numeric score/rank columns: leave as NaN (missing = "not ranked by this source", not a data error)
# This is intentional — do NOT median-fill these, it would misrepresent unranked universities as average performers

# Categorical/text columns: fill with 'Not Reported' instead of blank
text_cols = ['location', 'region', 'country_qs', 'country_times', 'country_code', 
             'qs_size_2025', 'qs_focus_2025', 'qs_status_2025',
             'qs_size_2024', 'qs_focus_2024', 'qs_status_2024']

for col in text_cols:
    df[col] = df[col].fillna('Not Reported')

print("Text columns filled. Remaining missing % (numeric columns, left intentional):")
print((df.isnull().mean() * 100).sort_values(ascending=False).head(15))

Text columns filled. Remaining missing % (numeric columns, left intentional):
shanghai_total_score                 73.391863
overall_score_2025                   45.452459
international_faculty_score_2024     35.318758
international_faculty_rank_2024      35.318758
sustainability_score_2024            34.670339
sustainability_rank_2024             34.670339
qs_research_2024                     34.658863
times_2023_rank                      34.561313
international_students_score_2024    34.509669
international_students_rank_2024     34.509669
faculty_student_score_2024           34.262925
citations_per_faculty_rank_2024      34.262925
employment_outcomes_rank_2024        34.262925
employment_outcomes_score_2024       34.262925
citations_per_faculty_score_2024     34.262925
dtype: float64


In [23]:
rank_cols = [c for c in df.columns if 'rank' in c]
for col in rank_cols:
    print(col, "-", df[col].dtype)

rank_2025 - str
rank_2024 - str
academic_reputation_rank_2025 - str
employer_reputation_rank_2025 - str
faculty_student_rank_2025 - str
citations_per_faculty_rank_2025 - str
international_faculty_rank_2025 - str
international_students_rank_2025 - str
international_research_network_rank_2025 - str
employment_outcomes_rank_2025 - str
sustainability_rank_2025 - str
times_2024_rank - str
times_2023_rank - str
academic_reputation_rank_2024 - str
employer_reputation_rank_2024 - str
faculty_student_rank_2024 - str
citations_per_faculty_rank_2024 - str
international_faculty_rank_2024 - str
international_students_rank_2024 - str
international_research_network_rank_2024 - str
employment_outcomes_rank_2024 - str
sustainability_rank_2024 - str
times_world_rank - str
shanghai_world_rank - str
national_rank - str


In [24]:
sample_cols = ['rank_2025', 'times_world_rank', 'shanghai_world_rank', 'national_rank']
for col in sample_cols:
    print(f"--- {col} ---")
    print(df[col].dropna().unique()[:20])
    print()

--- rank_2025 ---
<StringArray>
[      '306',       '113',       '144',   '671-680',   '661-670',   '621-630',
       '501',   '761-770',   '771-780',   '851-900',   '711-720',       '592',
       '308',       '481',       '477',   '631-640',     '1401+',       '552',
   '801-850', '1001-1200']
Length: 20, dtype: str

--- times_world_rank ---
<StringArray>
['301-350', '351-400', '201-250', '251-275', '251-300',     '167',     '125',
     '116',     '138',     '153',    '=106', '276-300', '601-800',     '147',
 '401-500', '501-600',     '161',     '127',     '148',     '146']
Length: 20, dtype: str

--- shanghai_world_rank ---
<StringArray>
['401-500', '301-400', '403-510', '402-503', '101-152', '102-150',      '93',
      '97',      '98',      '86',      '81',      '74',      '73', '101-150',
 '305-402', '303-401',      '88',     '100',      '96',      '94']
Length: 20, dtype: str

--- national_rank ---
<StringArray>
[    '5',   '3-5',     '4',   '4-6',   '4-5',     '2',   '2.0',   '4-

In [25]:
import re

def convert_rank(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()
    val = val.replace('=', '')       # remove tie marker, e.g. "=106" -> "106"
    val = val.replace('+', '')       # remove open-ended marker, e.g. "1401+" -> "1401"
    
    if '-' in val:
        parts = val.split('-')
        try:
            low, high = float(parts[0]), float(parts[1])
            return (low + high) / 2   # midpoint of the range
        except ValueError:
            return np.nan
    
    try:
        return float(val)
    except ValueError:
        return np.nan

rank_cols = [c for c in df.columns if 'rank' in c]

for col in rank_cols:
    df[col + '_numeric'] = df[col].apply(convert_rank)

print(df[[c for c in df.columns if 'rank' in c]].dtypes.tail(10))

faculty_student_rank_2024_numeric                   float64
citations_per_faculty_rank_2024_numeric             float64
international_faculty_rank_2024_numeric             float64
international_students_rank_2024_numeric            float64
international_research_network_rank_2024_numeric    float64
employment_outcomes_rank_2024_numeric               float64
sustainability_rank_2024_numeric                    float64
times_world_rank_numeric                            float64
shanghai_world_rank_numeric                         float64
national_rank_numeric                               float64
dtype: object


In [26]:
print(df[['rank_2025', 'rank_2025_numeric']].dropna().head(10))
print(df[['times_world_rank', 'times_world_rank_numeric']].dropna().head(10))

  rank_2025  rank_2025_numeric
0       306              306.0
1       306              306.0
2       306              306.0
3       306              306.0
4       306              306.0
5       306              306.0
6       306              306.0
7       306              306.0
8       306              306.0
9       306              306.0
  times_world_rank  times_world_rank_numeric
0          301-350                     325.5
1          301-350                     325.5
2          351-400                     375.5
3          351-400                     375.5
4          301-350                     325.5
5          301-350                     325.5
6          351-400                     375.5
7          351-400                     375.5
8          201-250                     225.5
9          201-250                     225.5


In [27]:
score_cols = [c for c in df.columns if 'score' in c]
for col in score_cols:
    if df[col].dtype == 'object':
        print(col, "-", df[col].dtype)

overall_score_2025 - object
academic_reputation_score_2024 - object
employer_reputation_score_2024 - object
faculty_student_score_2024 - object
citations_per_faculty_score_2024 - object
international_faculty_score_2024 - object
international_students_score_2024 - object
international_research_network_score_2024 - object
employment_outcomes_score_2024 - object
sustainability_score_2024 - object


In [29]:
check_cols = ['overall_score_2025', 'academic_reputation_score_2024', 'sustainability_score_2024']
for col in check_cols:
    print(f"--- {col} ---")
    print(df[col].dropna().unique()[:15])
    print()

--- overall_score_2025 ---
[35.7 57.9 53.3 24.1 21.0 35.3 25.0 25.1 22.5 51.0 44.8 54.1 40.6 24.8
 33.6]

--- academic_reputation_score_2024 ---
['20.0' '44.6' '55.8' '23.2' '9.5' '8.3' '11.5' '5.6' '9.2' '12.8' '22.4'
 '34.0' '36.1' '13.6' '15.5']

--- sustainability_score_2024 ---
['35.8' '95' '98.4' '1' '9.6' '11.8' '1.1' '2.5' '7' '30.5' '64.5' '3' '5'
 '1.6' '1.2']



In [30]:
score_cols_to_fix = ['overall_score_2025', 'academic_reputation_score_2024', 
                      'employer_reputation_score_2024', 'faculty_student_score_2024',
                      'citations_per_faculty_score_2024', 'international_faculty_score_2024',
                      'international_students_score_2024', 'international_research_network_score_2024',
                      'employment_outcomes_score_2024', 'sustainability_score_2024']

for col in score_cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[score_cols_to_fix].dtypes)

overall_score_2025                           float64
academic_reputation_score_2024               float64
employer_reputation_score_2024               float64
faculty_student_score_2024                   float64
citations_per_faculty_score_2024             float64
international_faculty_score_2024             float64
international_students_score_2024            float64
international_research_network_score_2024    float64
employment_outcomes_score_2024               float64
sustainability_score_2024                    float64
dtype: object


In [31]:
df.to_csv("university_cleaned.csv", index=False)
print("Saved successfully!")
print("Final shape:", df.shape)

Saved successfully!
Final shape: (17427, 105)
